# EffectLab — free temporary online engine

Run this notebook in Google Colab to start EffectLab's current Python backend in a temporary session. When it starts, copy the public API URL into EffectLab's **Engine URL** field. The URL changes each session, and the engine stops when this Colab session ends.

**Important:** this is a prototype model, not a realistic general-purpose body-swap model. Colab GPU access and session length are not guaranteed. Do not upload private or sensitive videos to a public tunnel.

In [ ]:
!git clone --depth 1 https://github.com/willoughbyorchard-cpu/effectlab-ai-video.git
%cd effectlab-ai-video
!pip -q install -r backend/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Setup complete. Next run the server cell.')

In [ ]:
import os, subprocess, time, re, threading
os.environ['EFFECTLAB_ALLOWED_ORIGINS'] = 'https://willoughbyorchard-cpu.github.io,http://localhost:7860,http://127.0.0.1:7860'
os.environ['PYTHONUNBUFFERED'] = '1'
server = subprocess.Popen(['python','-m','uvicorn','backend.app:app','--host','0.0.0.0','--port','7860'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(8)
print('Starting temporary secure tunnel...')
tunnel = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
deadline = time.time() + 60
while time.time() < deadline and public_url is None:
    line = tunnel.stdout.readline()
    if line:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            print('\nYour temporary EffectLab Engine URL: ' + public_url)
            print('Health check URL: ' + public_url + '/health')
            print('\nKeep this cell running. Copy the engine URL into EffectLab. It will stop when this session ends.')
            break
if public_url is None:
    print('Tunnel URL not detected. Check the output below, then rerun this cell if needed.')
    print(tunnel.stdout.read(2000))

## If it does not work
- Keep this notebook tab open and the server/tunnel cell running.
- Check that `https://YOUR-TUNNEL.trycloudflare.com/health` returns a small JSON health response.
- Colab may provide CPU rather than GPU; processing can be slow.
- The current EffectLab model is a small scratch-trained prototype and will not produce convincing high-detail body swaps.